# Trending commodity news: what is moving gold, silver, copper and oil right now?

**Powered by [Bigdata.com](https://bigdata.com)**

**The question:** *"Show me trending commodity headlines, such as gold or silver price up or down, where there is
significant interest and engagement around similar articles."*

This notebook finds the commodity stories that many outlets are covering **right now**, tells you whether each one
points **up or down** for the commodity's price, and flags early, strongly-worded stories that have not spread yet.

### What "trending" means here

Bigdata.com does **not** capture reader engagement (clicks, reads, shares). Trending is measured from
**publishing momentum**, which is a strong proxy for attention:

| Signal | Question it answers | Measured from |
|---|---|---|
| **Breadth** | How many articles and *distinct outlets* are covering the **same story**? (the "engagement around similar articles") | articles grouped into stories |
| **Acceleration** | Is this entity covered more than usual right now? (`spike = now ÷ normal`) | `/v1/search/volume` |
| **Recency** | How fresh is the latest coverage? | article timestamps |

A story needs **at least 2 independent outlets** to count as trending. Stories with strong positive or negative
sentiment from credible outlets that have *not yet* built wide coverage are shown separately as **Emerging**.

> If you have your own click or read data, it can be blended in later through the
> `external_engagement_score` hook in `trending/rank.py` (default 0).

### What this notebook does

| Step | What happens | Uses an AI model? |
|---|---|---|
| 1. Resolve | Commodity names → Bigdata.com entity IDs (Knowledge Graph); keyword mode if not found | No |
| 2. Momentum | Is the commodity covered more than usual right now? | No |
| 3. Fetch | Pull the current window's news for each commodity | No |
| 4. Relevance gate | Keep articles about the **market** (price, supply, demand, flows); drop miner promos and noise | Yes (falls back to rules) |
| 5. Group | Group articles that report the **same event** into one story | Yes (falls back to text similarity) |
| 6. Enrich | Title, summary, **direction (up / down / mixed)**, driver, materiality | Yes (falls back to keyword rules) |
| 7. Rank | Trend score per commodity, plus an **Emerging** list per commodity | No |
| 8. Report | HTML report grouped by commodity + CSV files in `outputs/` | No |

All counts, momentum figures and sentiment values are computed from Bigdata.com data. The AI model only filters,
groups, titles and labels articles. Run the cells top to bottom (about 1-2 minutes live, a few seconds in replay
mode). Each section shows its typical duration, and every code cell prints how long it took (`⏱`).

In [1]:
# Setup: make the shared `trending` package importable, tidy pandas output, time every cell.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from trending.timing import cell_timer
cell_timer.start()   # from here on, every cell ends with "⏱ <seconds>"; the last cell shows a summary

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from trending import config as cfg
from trending.pipeline import start, build_stories, finish, llm_label
from trending.resolve import resolve_all, entity_table
from trending.momentum import compute_momentum, momentum_table
from trending.rank import rank_trending, rank_emerging
from trending import render
from trending.pipeline import commodity_lists

⏱ 0.0 s


## 1. Configuration

Pick the commodities to track. Each one has a Knowledge Graph query and a few search phrases (`texts`) in
`cfg.COMMODITIES`; you can add your own (see the last section).

* **Run mode:** live when `BIGDATA_API_KEY` is set; otherwise the notebook replays the saved run in `fixtures/`.
* **LLM:** Anthropic Claude by default; OpenAI also supported (`LLM_PROVIDER=openai`); `cfg.USE_LLM = False` for none.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [2]:
# 1. Configuration and run mode
COMMODITIES = ["Gold", "Silver", "Copper", "Crude Oil"]   # also available: "Natural Gas", "Platinum", "Lithium"

cfg.WINDOW = "1D"          # "1D" = last 24h vs prior 7 days  |  "1W" = last 7 days vs prior 28 days
cfg.USE_LLM = True
cfg.GATE_KEEP = ["market_move"]   # add "company_specific" to also keep miner/producer company news

# Emerging list: commodity news is mostly neutral price reporting, which dilutes the *average* sentiment of a
# story. "max_abs" uses the strongest single passage instead, so sharp but quiet events stay visible.
cfg.SENTIMENT_MODE = "max_abs"
cfg.SENTIMENT_EXTREME = 0.5

cfg.REPLAY = cfg.REPLAY or not cfg.BIGDATA_API_KEY
client, llm, win = start("commodities")
selection = {name: cfg.COMMODITIES[name] for name in COMMODITIES}

Mode: LIVE · window 1D: 2026-10-12 22:40 → 2026-10-13 22:40 UTC
LLM: LLM(provider='openai', model='gpt-4.1')
⏱ 0.0 s


## 2. Resolve commodities (Knowledge Graph)

Each commodity is looked up among Knowledge Graph concepts of type *Commodity*. If one cannot be found, it runs in
**keyword mode**: articles are found by its search phrases instead of an entity filter, and momentum is measured the
same way. The `mode` column shows which was used.

> ⏱ **Typical time:** live under 1 s (a few seconds for names not seen before) · replay under 1 s

In [3]:
# 2. Resolve commodities
entities = resolve_all(client, commodities=selection, as_of=win.as_of)
entity_table(entities)

,input,label,entity_id,name,kg_type,kg_category,mode,how
0,gold,Gold,9768FE,Gold,Commodity,concepts,entity,KG concepts (Commodity)
1,silver,Silver,F96E34,Silver,Commodity,concepts,entity,KG concepts (Commodity)
2,copper,Copper,A4090F,Copper,Commodity,concepts,entity,KG concepts (Commodity)
3,crude oil,Crude Oil,FD9CFE,Crude Oil,Commodity,concepts,entity,KG concepts (Commodity)


⏱ 0.0 s


## 3. Momentum: which commodities are unusually loud right now?

`spike` = documents in the current window ÷ the commodity's normal volume for a window of the same length.
Above 1 means more coverage than usual. `trend` shows daily document counts for the last 8 days.

> ⏱ **Typical time:** live about 2 s · replay under 1 s

In [4]:
# 3. Momentum
def show_momentum(df):
    """Momentum table with a bar on the spike column (no extra dependencies)."""
    return (df.style
              .format({"spike": "{:.2f}×", "baseline_per_period": "{:,.0f}", "current_docs": "{:,}",
                       "sentiment": "{:+.2f}"}, na_rep="–")
              .bar(subset=["spike"], color="#f0a35e", vmin=0)
              .hide(axis="index"))

momentum = compute_momentum(client, entities, win)
show_momentum(momentum_table(momentum, entities))

entity,name,spike,current_docs,baseline_per_period,distinct_sources,sentiment,trend,mode
Copper,Copper,1.50×,374,250,143,+0.01,▄▅█▆▁▁▅▆,entity
Silver,Silver,1.41×,517,366,182,+0.04,▅▆▇█▂▁▅▇,entity
Crude Oil,Crude Oil,1.27×,"2,087","1,644",564,-0.28,▅▅█▅▁▁▆▅,entity
Gold,Gold,1.21×,"1,747","1,448",469,-0.00,▇█▇▇▁▁▅▆,entity


⏱ 2.1 s


## 4. Relevance gate, story grouping, enrichment and ranking

Keyword and entity searches for commodities also return company promotions ("Can X Gold strengthen its exploration
story?") and unrelated uses of the word. The **relevance gate** labels every article as `market_move`,
`company_specific` or `irrelevant` and keeps `market_move` by default. The table below shows how many articles fell
into each class. Then articles are grouped into stories, the leading stories are enriched (direction is required
here), and everything is scored. This is the longest step: most of it is the AI model.

> ⏱ **Typical time:** live about 1-2 min · replay under 1 s

In [5]:
# 4. Relevance gate, grouping, enrichment and ranking
stories, articles, gate_stats = build_stories(client, llm, entities, win, momentum, gate=True,
                                              per_group=cfg.TOP_N_PER_GROUP, commodity=True)
keys = [e.key for e in entities]
combined, by_commodity, emerging_by_commodity = commodity_lists(stories, keys, win, use_llm=llm is not None)

display(pd.DataFrame(gate_stats).T.fillna(0).astype(int).rename_axis("relevance gate"))

cols = ["rank", "title", "entities", "direction", "driver", "articles", "sources", "spike", "materiality",
        "avg_sentiment", "trend_score"]
trending_df = render.stories_frame(combined, "trending")
print("Top trending across commodities")
trending_df[cols]

Step 3  fetched 743 unique articles: Gold 256, Silver 109, Copper 101, Crude Oil 277  (5s)


        relevance gate kept 666 articles (market_move)  (12s)


Step 4  grouped into 349 stories, 82 covered by ≥2 sources  (90s)


        merged 12 duplicate stories among the top 60


Step 5  enriched 62 stories (title, summary, direction, driver, materiality)  (96s)
Step 6  scored all stories


,market_move,company_specific,irrelevant
relevance gate,,,
Gold,231,25,0
Silver,87,14,8
Copper,84,17,0
Crude Oil,264,9,4


Top trending across commodities


,rank,title,entities,direction,driver,articles,sources,spike,materiality,avg_sentiment,trend_score
0,1,Brent Crude Oil Prices Fall After Previous Gains,Crude Oil,down,Russian diesel supply,43,37,1.27,4,-0.422,2.2195
1,2,"Oil Prices End Mixed Amid Russian Diesel Deal, Middle East Tensions",Crude Oil,mixed,"Russian diesel deal, Middle East conflic",41,27,1.27,4,-0.405,2.1892
2,3,"OPEC Raises 2027 Oil Demand Forecast, Cuts 2026 Outlook",Crude Oil,up,OPEC demand forecast,30,27,1.27,4,-0.217,2.1008
3,4,WTI Crude Rises on Middle East Supply Concerns,Crude Oil,up,Middle East tensions,26,23,1.27,4,-0.286,2.0123
4,5,Saudi Arabia Leads OPEC+ Oil Output Surge in September,Crude Oil,up,Saudi production rebound,16,15,1.27,4,-0.117,1.7784
5,6,Gold and Silver Prices Fall Ahead of US Inflation Data,"Silver, Gold",down,"US dollar strength, CPI anticipation",19,12,1.41,3,-0.210,1.7260
6,7,Central Banks Reduce Gold Reserves Amid Shifting Market Dynamics,Gold,down,central bank selling,11,8,1.21,3,-0.227,1.4545
7,8,WTI Oil Falls as Russian Diesel Supply Offsets Geopolitical Risks,Crude Oil,down,Russian diesel supply,9,7,1.27,3,-0.404,1.3714
8,9,Tight Supply Drives October Price Hikes for Copper-Related Electronic Materials,Copper,up,tight supply-demand,7,6,1.50,4,-0.001,1.3076
9,10,Silver Market Deficit to Widen for Sixth Consecutive Year,Silver,mixed,"supply deficit, Fed policy",6,6,1.41,4,-0.342,1.2852


⏱ 1 min 36 s


### Trending stories per commodity

Each commodity's top stories, with the direction the story implies for the commodity's price.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [6]:
# 4b. Trending stories per commodity
for key, ss in by_commodity.items():
    m = momentum.get(key)
    print(f"\n{key}  ·  {m.spike:.1f}× usual coverage" if m else f"\n{key}")
    for i, s in enumerate(ss, 1):
        arrow = {"up": "▲", "down": "▼", "mixed": "◆"}.get(s.direction, "·")
        print(f"  {i}. {arrow} {s.direction:<5} {s.title}  [{s.articles} articles · {s.sources} sources · {s.driver}]")


Gold  ·  1.2× usual coverage
  1. ▼ down  Gold and Silver Prices Fall Ahead of US Inflation Data  [19 articles · 12 sources · US dollar strength, CPI anticipation]
  2. ▼ down  Central Banks Reduce Gold Reserves Amid Shifting Market Dynamics  [11 articles · 8 sources · central bank selling]
  3. ▼ down  Stronger Rupee Pressures Indian Gold Prices Lower for Second Day  [6 articles · 6 sources · rupee appreciation]
  4. ▲ up    Gold Rises Despite Trump-Announced Energy Ceasefire  [6 articles · 5 sources · energy ceasefire news]
  5. ▼ down  Fed Tightening Expectations Push New York Gold Prices Lower  [5 articles · 5 sources · Fed tightening expectations]

Silver  ·  1.4× usual coverage
  1. ▼ down  Gold and Silver Prices Fall Ahead of US Inflation Data  [19 articles · 12 sources · US dollar strength, CPI anticipation]
  2. ◆ mixed Silver Market Deficit to Widen for Sixth Consecutive Year  [6 articles · 6 sources · supply deficit, Fed policy]
  3. ▲ up    Silver Prices Rise Slightly Desp

## 5. Emerging signals per commodity: strong sentiment, limited coverage so far

Stories that are **not** in Trending but carry strongly positive or negative Bigdata.com sentiment from credible
outlets (a single source is allowed only for top-tier outlets, rank 1-2), have materiality 4-5, and point in a clear
direction. For example, a sharp negative silver supply story covered by one tier-1 outlet.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [7]:
# 5. Emerging signals per commodity
emerging = [s for k in keys for s in emerging_by_commodity[k]]
emerging = list({s.sid: s for s in emerging}.values())          # a story about two commodities counts once
emerging_df = render.stories_frame(emerging, "emerging")
if len(emerging_df):
    display(emerging_df[["title", "entities", "direction", "sentiment_signal", "sources", "best_source_rank",
                         "representative_source", "driver", "materiality", "emerging_score"]])
else:
    print("No Emerging stories in this window. Try cfg.SENTIMENT_EXTREME = 0.35 (see README, Tuning).")

,title,entities,direction,sentiment_signal,sources,best_source_rank,representative_source,driver,materiality,emerging_score
0,Gold Analysts Predict Potential Rally Toward $5400 Amid Buy Signals,Gold,up,0.71,2,3,Gold-Eagle.com,technical signals,4,0.5599
1,Gold Falls Over 25% Despite Inflation and Middle East Tensions,Gold,down,-0.66,2,3,Forbes Argentina,Fed rate hikes,4,0.5203
2,Gold Slides as US Yields Surge Ahead of CPI Data,Gold,down,-0.63,2,1,Fxstreet,US yields surge,4,0.3335
3,Gold rebounds from early dip ahead of key US CPI data,Gold,up,0.76,2,2,Flush Financial Services Network,CPI anticipation,4,0.1537
4,Gold slips as Middle East conflict fuels inflation and rate hike fears,Gold,down,-0.62,1,2,Yahoo! Finance,rate hike concerns,4,0.1400
5,Silver rises amid Middle East tensions and stalled US-Iran talks,Silver,up,-0.65,2,2,TechJuice,geopolitical tensions,4,0.1449
6,Copper Prices Surge Amid Supply Risks and Labor Disputes,Copper,up,-0.66,1,2,Flush Financial Services Network,supply constraints,5,0.3440
7,"Gold Prices Face Mixed Pressures from Dollar, Oil, and Central Bank Buying",Copper,mixed,-0.61,1,1,FXStreet News,dollar strength and central bank buying,4,0.2097
8,"Oil Steadies After Three-Day Rally Amid Mideast, Russia Tensions",Crude Oil,mixed,-0.61,2,2,Livemint,geopolitical tensions,4,0.4684
9,Iran Ship Attacks Threaten Oil Exports Through Strait of Hormuz,Crude Oil,up,-0.60,1,1,TBS NEWS DIG,geopolitical risk,5,0.3920


⏱ 0.0 s


## 6. The report

Grouped by commodity: the header shows the momentum and the net direction of the commodity's top stories
(e.g. "Gold ↑ 2.9× · mostly up"); each commodity lists its Trending stories, then its Emerging items. A combined
"Top trending across commodities" list sits at the top. Saved to `outputs/` as HTML and CSV.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [8]:
# 6. Build and save the report
ctx = render.ReportContext(win=win, momentum=momentum, show_direction=True, names={e.key: e.name for e in entities})
TITLE = "Trending commodity news"
gate_note = (f"Relevance gate kept {sum(len(v) for v in articles.values()):,} market-moving articles "
             f"({', '.join(cfg.GATE_KEEP)}); company promotions and unrelated articles were removed.")
report = render.commodities_report(title=TITLE, scope=", ".join(COMMODITIES), ctx=ctx, combined=combined,
                                   groups=by_commodity, emerging_by_group=emerging_by_commodity,
                                   llm_label=llm_label(llm), gate_note=gate_note)
paths = render.save_outputs(report, TITLE, {"trending": trending_df, "emerging": emerging_df},
                            "commodities", day=win.as_of)
for kind, p in paths.items():
    print(f"saved {kind:<9} → {os.path.relpath(p, ROOT)}")
display(HTML(report))

saved html      → outputs/commodities_trending_2026-10-13.html
saved trending  → outputs/commodities_2026-10-13_trending.csv
saved emerging  → outputs/commodities_2026-10-13_emerging.csv


⏱ 0.0 s


In [9]:
# Save this run's responses to fixtures/ (offline replay)
finish(client, llm)

Bigdata.com query units used this session: 136.9, LLM calls 30 (210 s summed over parallel calls, 0 from cache)
Fixtures saved under fixtures/commodities/ (removed 41 stale files).
⏱ 0.0 s


## Run time

Every code cell above ends with `⏱`: how long it took in this run. The table adds them up per step. Live runs are
dominated by the AI steps (grouping and enrichment) and by Bigdata.com calls; replay runs take a few seconds.

In [10]:
# Run time: how long each step took in this run
cell_timer.summary()

,step,time,share,seconds
0,Setup,0.0 s,0%,0.0
1,1. Configuration and run mode,0.0 s,0%,0.0
2,2. Resolve commodities,0.0 s,0%,0.0
3,3. Momentum,2.1 s,2%,2.1
4,"4. Relevance gate, grouping, enrichment and ranking",1 min 36 s,98%,95.7
5,4b. Trending stories per commodity,0.0 s,0%,0.0
6,5. Emerging signals per commodity,0.0 s,0%,0.0
7,6. Build and save the report,0.0 s,0%,0.0
8,Save this run's responses to fixtures/ (offline replay),0.0 s,0%,0.0
9,Total,1 min 38 s,100%,97.8


## 7. How to adapt

| To… | Change |
|---|---|
| Track other commodities | Add them to `COMMODITIES` above; define new ones in `cfg.COMMODITIES`, e.g. `cfg.COMMODITIES["Iron Ore"] = {"kg_query": "iron ore", "texts": ["iron ore price", "iron ore China steel demand"]}` |
| Change what counts as relevant | `cfg.GATE_KEEP = ["market_move", "company_specific"]` to keep company news too |
| Steer what is retrieved | edit each commodity's `texts` (search phrases, max 5) |
| Look at the past week | `cfg.WINDOW = "1W"` |
| Make Emerging stricter / looser | `cfg.SENTIMENT_EXTREME` (e.g. 0.35 for more items), `cfg.SENTIMENT_MODE` (`"mean"` or `"max_abs"`), `cfg.EMERGING_MIN_MATERIALITY` |
| Show more stories per commodity | `cfg.TOP_N_PER_GROUP`, `cfg.COMBINED_TOP_N` |
| Change ranking | `cfg.WEIGHTS`, `cfg.TRENDING_MIN_SOURCES` |
| Blend in your own engagement data | `external_engagement_score(story)` in `trending/rank.py` |

See `README.md` for the full list of settings and the limitations.